# **Assignment 09: Framepack**

- Due: 06November2025

**Details**
- Dataset: https://www.kaggle.com/datasets/sharjeelmazhar/human-activity-recognition-video-dataset
- Use FVD as metric: https://openreview.net/pdf?id=rylgEULtdNLinks to an external site.​
- Run Wan2.1 (from previous assignment) to generate video from image​
- Use starting frame of each video, 10 per category​
- Run framepack to generate video from image​
- Same starting frame and same 10​
- Benchmark the two of them with FVD​
- Report, Code, Video​

## Setup

In [1]:
## Import Libraries

# Set CUDA_VISIBLE_DEVICES to make both GPUs visible
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

# Install all packages for from the requirements.txt
%pip install -r requirements.txt

import torch
import torch.nn as nn
import torchvision
import datasets
import cv2
import matplotlib.pyplot as plt
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch import optim
from tqdm.notebook import tqdm
from torchinfo import summary
import einops
import PIL
import numpy as np
import pandas as pd
# Use a pipeline as a high-level helper
from transformers import pipeline
import time
import psutil
import gc
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import json
from collections import defaultdict
import numpy as np

# Authorize Huggingface account
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv('/mnt/Storage02/SoftwareDev/CAP_6411_Assignments/.env')

# Get Hugging Face token
hf_token = os.getenv('HUGGINGFACE_HUB_TOKEN') or os.getenv('HF_TOKEN')

if hf_token:
    print("Found Hugging Face token in environment variables")
    
    
    from huggingface_hub import login, whoami
    
    try:
        # Login to Hugging Face Hub
        login(token=hf_token)
        
        # Verify login by getting user info
        user_info = whoami()
        print(f"Successfully authenticated with Hugging Face!")
        print(f"Logged in as: {user_info['name']}")
        
        # Set the token as environment variable for other libraries
        os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token
        os.environ['HF_TOKEN'] = hf_token
        
    except Exception as e:
        print(f"Authentication failed: {e}")
        print("Will proceed without pre-trained models if needed")
        hf_token = None
else:
    print("No Hugging Face token found in .env file")
    print("Please add HUGGINGFACE_HUB_TOKEN=your_token_here to your .env file")
    hf_token = None


# If no logs folder exists, create one
if not os.path.exists("logs"):
    os.makedirs("logs")

# If no checkpoints folder exists, create one
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints")

# If no data folder exists, create one
if not os.path.exists("data"):
    os.makedirs("data")


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Found Hugging Face token in environment variables
Found Hugging Face token in environment variables


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Successfully authenticated with Hugging Face!
Logged in as: malneyugnfl


In [2]:
# GPU Setup 
# Comprehensive GPU diagnostics
print("\n=== GPU Diagnostics ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs detected: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print("\n=== All Available GPUs ===")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}:")
        print(f"  Name: {props.name}")
        print(f"  Total Memory: {props.total_memory / 1024**3:.2f} GB")
        print(f"  Multi-processor count: {props.multi_processor_count}")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print()

# Device selection with preference for cuda:1 (A6000) -> cuda:0 (4090) -> mps (Apple Silicon) -> cpu
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    device = torch.device('cuda:1')  # This should now be your A6000!
    print(f"Using GPU 1: {torch.cuda.get_device_name(1)}")
elif torch.cuda.is_available():
    device = torch.device('cuda:0')
    print(f"Using GPU 0: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using Apple Silicon MPS")
else:
    device = torch.device('cpu')
    print("Using CPU")

print(f"Selected device: {device}")

# If no logs folder exists, create one
if not os.path.exists("logs"):
    os.makedirs("logs")

# If no checkpoints folder exists, create one
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints")

# If no data folder exists, create one
if not os.path.exists("data"):
    os.makedirs("data")

def get_memory_usage():
    """Get current memory usage in MB"""
    process = psutil.Process()
    return process.memory_info().rss / 1024 / 1024

def get_gpu_memory_usage():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024 / 1024
    elif device.type == 'mps':
        # MPS doesn't have direct memory monitoring like CUDA
        # Return 0 as a placeholder
        return 0
    return 0


=== GPU Diagnostics ===
PyTorch version: 2.9.0+cu128
CUDA available: True
MPS available: False
CUDA version: 12.8
Number of GPUs detected: 2

=== All Available GPUs ===
GPU 0:
  Name: NVIDIA GeForce RTX 4090 Laptop GPU
  Total Memory: 15.70 GB
  Multi-processor count: 76
  Compute Capability: 8.9

GPU 1:
  Name: NVIDIA RTX A6000
  Total Memory: 47.53 GB
  Multi-processor count: 84
  Compute Capability: 8.6

Using GPU 1: NVIDIA RTX A6000
Selected device: cuda:1


In [3]:
# Function to clear memory for both CUDA and MPS
def clear_memory():
    """Clear CPU and GPU memory"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()
    print(f"Cleared memory. Current CPU memory usage: {get_memory_usage():.2f} MB, GPU memory usage: {get_gpu_memory_usage():.2f} MB")

## Import WAN Model 

In [ ]:
## Source: https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B

import os
import shutil

# Clone the WAN 2.1 repository only if it doesn't already exist
if not os.path.exists('Wan2.1'):
    print("Cloning WAN 2.1 repository...")
    !git clone https://github.com/Wan-Video/Wan2.1.git
    
    # Deactivate the cloned repository by removing .git folder
    git_folder = os.path.join('Wan2.1', '.git')
    if os.path.exists(git_folder):
        print("Deactivating cloned repository (removing .git folder)...")
        shutil.rmtree(git_folder)
        print("Wan2.1 is no longer a git repository")
    else:
        print("No .git folder found in cloned repository")
else:
    print("WAN 2.1 repository already exists, skipping clone.")

# Change to the Wan2.1 directory
os.chdir('Wan2.1')
print(f"Changed to directory: {os.getcwd()}")

# Install requirements (ensure torch >= 2.4.0)
!pip install -r requirements.txt

# Download the model from Hugging Face only if it doesn't already exist
if not os.path.exists('./Wan2.1-T2V-1.3B'):
    print("Downloading WAN 2.1 model...")
    !huggingface-cli download Wan-AI/Wan2.1-T2V-1.3B --local-dir ./Wan2.1-T2V-1.3B
else:
    print("WAN 2.1 model already exists, skipping download.")

In [ ]:
Human_Action_Categories = [
    "Human Clapping",
    "Human Meet and Split",
    "Human Sitting",
    "Human Standing Still",
    "Human Walking",
    "Human Walking While Reading Book",
    "Human Walking While Using Phone"
]

In [ ]:
# Generate videos for each Human Action Category using WAN 2.1
import subprocess
import os
import shutil
import glob
from datetime import datetime

# Get the current working directory
current_dir = os.getcwd()
print(f"Current directory: {current_dir}")

# Determine the Wan2.1 directory path
if current_dir.endswith('Wan2.1'):
    # Already in Wan2.1 directory
    wan_dir = current_dir
    print(f"Already in Wan2.1 directory: {wan_dir}")
elif os.path.exists(os.path.join(current_dir, 'Wan2.1')):
    # Wan2.1 is a subdirectory of current directory
    wan_dir = os.path.join(current_dir, 'Wan2.1')
    print(f"Found Wan2.1 as subdirectory: {wan_dir}")
else:
    # Try to find Wan2.1 in parent directory
    parent_dir = os.path.dirname(current_dir)
    if os.path.exists(os.path.join(parent_dir, 'Wan2.1')):
        wan_dir = os.path.join(parent_dir, 'Wan2.1')
        print(f"Found Wan2.1 in parent directory: {wan_dir}")
    else:
        raise FileNotFoundError("Could not locate Wan2.1 directory. Please check the path.")

# Verify generate.py exists
generate_script = os.path.join(wan_dir, 'generate.py')
if not os.path.exists(generate_script):
    raise FileNotFoundError(f"generate.py not found at {generate_script}")

print(f"Using Wan2.1 directory: {wan_dir}")
print(f"Found generate.py: {generate_script}")

# Create output directory for generated videos
output_dir = os.path.join(os.path.dirname(wan_dir), 'generated_videos', 'wan2.1')
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}\n")

# Generate videos for each category
for idx, category in enumerate(Human_Action_Categories):
    print(f"\n{'='*60}")
    print(f"Generating video {idx+1}/{len(Human_Action_Categories)}: {category}")
    print(f"{'='*60}\n")
    
    # Create a descriptive prompt for each category
    prompt = f"A person {category.lower().replace('human ', '')} in a clear indoor environment"
    
    # Get list of existing video files before generation
    existing_videos = set(glob.glob(os.path.join(wan_dir, '*.mp4')))
    
    # Build the command
    cmd = [
        "python", "generate.py",
        "--task", "t2v-1.3B",
        "--size", "832*480",
        "--ckpt_dir", "./Wan2.1-T2V-1.3B",
        "--sample_shift", "8",
        "--sample_guide_scale", "6",
        "--prompt", prompt
    ]
    
    # Set environment variable for GPU
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = '1'
    
    try:
        # Run the command from the Wan2.1 directory
        result = subprocess.run(
            cmd,
            cwd=wan_dir,
            env=env,
            capture_output=True,
            text=True,
            check=True
        )
        
        print(f"✓ Successfully generated video for: {category}")
        print(f"Prompt used: {prompt}")
        
        # Find the newly generated video file
        new_videos = set(glob.glob(os.path.join(wan_dir, '*.mp4'))) - existing_videos
        
        if new_videos:
            # Get the most recent video file
            latest_video = max(new_videos, key=os.path.getctime)
            
            # Create a clean filename from the category name
            clean_category = category.replace(' ', '_').replace('/', '_')
            new_filename = f"{clean_category}_{idx+1:02d}.mp4"
            destination = os.path.join(output_dir, new_filename)
            
            # Move the video to the output directory
            shutil.move(latest_video, destination)
            print(f"✓ Saved video to: {destination}")
        else:
            print(f"⚠ Warning: Could not find generated video file for {category}")
        
        # Print any output from the command
        if result.stdout:
            print(f"Output: {result.stdout[-200:]}")  # Last 200 chars
            
    except subprocess.CalledProcessError as e:
        print(f"✗ Error generating video for: {category}")
        print(f"Error: {e.stderr}")
        continue
    
    print(f"\nCompleted {idx+1}/{len(Human_Action_Categories)} videos")

print(f"\n{'='*60}")
print("All video generation tasks completed!")
print(f"Videos saved to: {output_dir}")
print(f"{'='*60}")

# List all generated videos
print("\nGenerated videos:")
for video_file in sorted(glob.glob(os.path.join(output_dir, '*.mp4'))):
    print(f"  - {os.path.basename(video_file)}")

## Get first frame from Generated Wan 2.1 Videos

In [ ]:
# Extract first and last frame from each generated WAN 2.1 video
import cv2
import os
import glob

# Define paths
wan_videos_dir = os.path.join('generated_videos', 'wan2.1')
first_frames_dir = os.path.join('generated_videos', 'FirstVideoFrame')
last_frames_dir = os.path.join('generated_videos', 'LastVideoFrame')

# Create the directories if they don't exist
os.makedirs(first_frames_dir, exist_ok=True)
os.makedirs(last_frames_dir, exist_ok=True)

# Get all mp4 files in the wan2.1 directory
video_files = sorted(glob.glob(os.path.join(wan_videos_dir, '*.mp4')))

print(f"Found {len(video_files)} videos in {wan_videos_dir}")
print(f"Extracting first frames to {first_frames_dir}")
print(f"Extracting last frames to {last_frames_dir}\n")

# Process each video
for video_path in video_files:
    # Get the video filename without extension
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    
    # Open the video
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"✗ Error: Could not open video {video_name}")
        continue
    
    # Get total number of frames
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Read the first frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    ret_first, first_frame = cap.read()
    
    # Read the last frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, total_frames - 1)
    ret_last, last_frame = cap.read()
    
    if ret_first:
        # Save the first frame as JPG
        first_output_path = os.path.join(first_frames_dir, f"{video_name}_frame0.jpg")
        cv2.imwrite(first_output_path, first_frame)
        print(f"✓ Saved first frame: {os.path.basename(first_output_path)}")
    else:
        print(f"✗ Error: Could not read first frame from {video_name}")
    
    if ret_last:
        # Save the last frame as JPG
        last_output_path = os.path.join(last_frames_dir, f"{video_name}_frame{total_frames-1}.jpg")
        cv2.imwrite(last_output_path, last_frame)
        print(f"✓ Saved last frame: {os.path.basename(last_output_path)}")
    else:
        print(f"✗ Error: Could not read last frame from {video_name}")
    
    print(f"  (Total frames: {total_frames})\n")
    
    # Release the video capture
    cap.release()

print(f"\n{'='*60}")
print(f"Extraction complete!")
print(f"First frames saved to: {first_frames_dir}")
print(f"Last frames saved to: {last_frames_dir}")
print(f"{'='*60}")

# List all extracted frames
print("\nFirst frames:")
first_frame_files = sorted(glob.glob(os.path.join(first_frames_dir, '*.jpg')))
for frame_file in first_frame_files:
    print(f"  - {os.path.basename(frame_file)}")

print("\nLast frames:")
last_frame_files = sorted(glob.glob(os.path.join(last_frames_dir, '*.jpg')))
for frame_file in last_frame_files:
    print(f"  - {os.path.basename(frame_file)}")

## Import and Use Framepack
Source: https://huggingface.co/docs/diffusers/main/api/pipelines/framepack


In [ ]:
import torch
from diffusers import HunyuanVideoFramepackPipeline, HunyuanVideoFramepackTransformer3DModel
from diffusers.utils import export_to_video, load_image
from transformers import SiglipImageProcessor, SiglipVisionModel

# Single-GPU setup: Use A6000 (cuda:1) with 48GB VRAM
# This pipeline doesn't support multi-GPU splitting, so we use one powerful GPU
print(f"Using device: {device} ({torch.cuda.get_device_name(device)})")
print(f"Available VRAM: {torch.cuda.get_device_properties(device).total_memory / 1024**3:.2f} GB")

# Clear any existing GPU memory
torch.cuda.empty_cache()
import gc
gc.collect()

# Load all components on the same device (cuda:1 - A6000)
print("\nLoading Framepack components on A6000 (cuda:1)...")
print("Loading Transformer...")
transformer = HunyuanVideoFramepackTransformer3DModel.from_pretrained(
    "lllyasviel/FramePackI2V_HY", torch_dtype=torch.bfloat16
).to(device)

print("Loading Feature Extractor and Image Encoder...")
feature_extractor = SiglipImageProcessor.from_pretrained(
    "lllyasviel/flux_redux_bfl", subfolder="feature_extractor"
)
image_encoder = SiglipVisionModel.from_pretrained(
    "lllyasviel/flux_redux_bfl", subfolder="image_encoder", torch_dtype=torch.float16
).to(device)

print("Creating pipeline...")
pipe = HunyuanVideoFramepackPipeline.from_pretrained(
    "hunyuanvideo-community/HunyuanVideo",
    transformer=transformer,
    feature_extractor=feature_extractor,
    image_encoder=image_encoder,
    torch_dtype=torch.float16,
)

# Move entire pipeline to the same device
pipe = pipe.to(device)

# Enable memory optimizations
print("Enabling memory optimizations...")
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()

print("\n=== Model Configuration ===")
print(f"All components on: {device}")
print(f"Transformer: {transformer.device}")
print(f"Image Encoder: {image_encoder.device}")
print(f"VAE: {pipe.vae.device}")
print(f"VAE tiling: Enabled")
print(f"VAE slicing: Enabled")
print("="*30)

print("="*30)

prompt = "CG animation style, a small blue bird takes off from the ground, flapping its wings. The bird's feathers are delicate, with a unique pattern on its chest. The background shows a blue sky with white clouds under bright sunshine. The camera follows the bird upward, capturing its flight and the vastness of the sky from a close-up, low-angle perspective."
first_image = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/flf2v_input_first_frame.png"
)
last_image = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/flf2v_input_last_frame.png"
)

print("\nGenerating video...")
# Generate with latent output to avoid OOM during VAE decoding
result = pipe(
    image=first_image,
    last_image=last_image,
    prompt=prompt,
    height=512,
    width=512,
    num_frames=91,
    num_inference_steps=30,
    guidance_scale=9.0,
    generator=torch.Generator(device=device).manual_seed(0),
    sampling_type="inverted_anti_drifting",
    output_type="latent",  # Get latents instead of decoded frames to save memory
)

print("✓ Latent generation complete!")
print("Decoding latents to frames in batches...")

# Move latents to CPU and decode in smaller batches to avoid OOM
latents = result.frames[0].cpu()
decoded_frames = []

# Decode in small batches
batch_size = 10  # Decode 10 frames at a time
num_frames = latents.shape[0]

for i in range(0, num_frames, batch_size):
    end_idx = min(i + batch_size, num_frames)
    print(f"  Decoding frames {i+1}-{end_idx}/{num_frames}...")
    
    # Move batch back to GPU for decoding
    batch_latents = latents[i:end_idx].to(device)
    
    # Decode batch
    with torch.no_grad():
        decoded_batch = pipe.vae.decode(batch_latents / pipe.vae.config.scaling_factor, return_dict=False)[0]
    
    # Move decoded frames to CPU immediately
    decoded_batch = decoded_batch.cpu()
    decoded_frames.append(decoded_batch)
    
    # Clear GPU cache after each batch
    torch.cuda.empty_cache()

# Concatenate all batches
import torch
output = torch.cat(decoded_frames, dim=0)

print("✓ All frames decoded!")
print(f"Output shape: {output.shape}")

# Convert to format expected by export_to_video
# Assuming output is in shape [frames, channels, height, width]
output = (output / 2 + 0.5).clamp(0, 1)  # Normalize to [0, 1]
output = (output * 255).to(torch.uint8)  # Convert to uint8
output = output.permute(0, 2, 3, 1).cpu().numpy()  # [frames, height, width, channels]

print("Exporting video...")
export_to_video(output, "output.mp4", fps=30)
print("✓ Video saved to output.mp4")

Using device: cuda:1 (NVIDIA RTX A6000)
Available VRAM: 47.53 GB

Loading Framepack components on A6000 (cuda:1)...
Loading Transformer...


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading Feature Extractor and Image Encoder...
Creating pipeline...
Creating pipeline...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Enabling memory optimizations...

=== Model Configuration ===
All components on: cuda:1
Transformer: cuda:1
Image Encoder: cuda:1
VAE: cuda:1
VAE tiling: Enabled
VAE slicing: Enabled

Generating video...

Generating video...


Token indices sequence length is longer than the specified maximum sequence length for this model (78 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['.']
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['.']


  0%|          | 0/30 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 14.34 GiB. GPU 1 has a total capacity of 47.53 GiB of which 2.81 GiB is free. Including non-PyTorch memory, this process has 44.71 GiB memory in use. Of the allocated memory 41.93 GiB is allocated by PyTorch, and 2.46 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)